# Spaceship Titanic - Cleaning and Parsing Key Columns

This notebook cleans and parses important columns in `train.csv` and `test.csv` for modeling.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
base_dir = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
train = pd.read_csv(base_dir / 'train.csv')
test = pd.read_csv(base_dir / 'test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (8693, 14)
Test shape: (4277, 13)


## Cleaning and Parsing Function

In [3]:
def clean_and_parse(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # -----------------------------
    # 1) PassengerId parsing
    # -----------------------------
    pid_split = df['PassengerId'].str.split('_', n=1, expand=True)
    df['GroupId'] = pid_split[0]
    df['GroupMemberId'] = pd.to_numeric(pid_split[1], errors='coerce')

    # -----------------------------
    # 2) Cabin parsing: deck/num/side
    # -----------------------------
    cabin_split = df['Cabin'].fillna('Unknown/Unknown/Unknown').str.split('/', n=2, expand=True)
    df['CabinDeck'] = cabin_split[0]
    df['CabinNum'] = pd.to_numeric(cabin_split[1], errors='coerce')
    df['CabinSide'] = cabin_split[2]

    # -----------------------------
    # 3) Name parsing
    # -----------------------------
    name_split = df['Name'].fillna('Unknown Unknown').str.split(' ', n=1, expand=True)
    df['FirstName'] = name_split[0]
    df['LastName'] = name_split[1].fillna('Unknown')

    # -----------------------------
    # 4) Boolean cleanup for CryoSleep and VIP
    # -----------------------------
    # Keep as nullable boolean first, then fill missing with False for model-ready baseline
    df['CryoSleep'] = df['CryoSleep'].astype('boolean').fillna(False).astype(bool)
    df['VIP'] = df['VIP'].astype('boolean').fillna(False).astype(bool)

    # -----------------------------
    # 5) Numeric cleanup for Age and spend columns
    # -----------------------------
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Age'] = df['Age'].fillna(df['Age'].median())

    # -----------------------------
    # 6) Optional helpful derived features
    # -----------------------------
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)

    # Count group size from parsed GroupId
    df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')

    return df

## Apply Cleaning

In [4]:
train_clean = clean_and_parse(train)
test_clean = clean_and_parse(test)

print('Train cleaned shape:', train_clean.shape)
print('Test cleaned shape:', test_clean.shape)

Train cleaned shape: (8693, 24)
Test cleaned shape: (4277, 23)


## Quick Sanity Checks

In [5]:
new_cols = ['GroupId', 'GroupMemberId', 'CabinDeck', 'CabinNum', 'CabinSide', 'FirstName', 'LastName', 'TotalSpend', 'NoSpend', 'GroupSize']
print('Parsed/engineered columns present in train:', all(c in train_clean.columns for c in new_cols))
print('Parsed/engineered columns present in test:', all(c in test_clean.columns for c in new_cols))

train_clean[new_cols].head()

Parsed/engineered columns present in train: True
Parsed/engineered columns present in test: True


,GroupId,GroupMemberId,CabinDeck,CabinNum,CabinSide,FirstName,LastName,TotalSpend,NoSpend,GroupSize
0,0001,1,B,0.0,P,Maham,Ofracculy,0.0,1,1
1,0002,1,F,0.0,S,Juanna,Vines,736.0,0,1
2,0003,1,A,0.0,S,Altark,Susent,10383.0,0,2
3,0003,2,A,0.0,S,Solam,Susent,5176.0,0,2
4,0004,1,F,1.0,S,Willy,Santantines,1091.0,0,1


In [6]:
# See remaining missing values after baseline cleaning
train_clean.isna().sum().sort_values(ascending=False).head(20)

HomePlanet       201
Name             200
Cabin            199
CabinNum         199
Destination      182
GroupId            0
NoSpend            0
TotalSpend         0
LastName           0
FirstName          0
CabinSide          0
CabinDeck          0
GroupMemberId      0
PassengerId        0
Transported        0
VRDeck             0
Spa                0
ShoppingMall       0
FoodCourt          0
RoomService        0
dtype: int64

## Optional: Save Cleaned Datasets

In [ ]:
# Uncomment to save cleaned files
# train_clean.to_csv(base_dir / 'train_clean.csv', index=False)
# test_clean.to_csv(base_dir / 'test_clean.csv', index=False)